# Aviation Trends

This notebook analyses daily aviation data to and from - Dubai, Doha, Amman, Cairo, Casablanca, Riyadh, Jeddah and Abu Dhabi. The main analyses involves daily and weekly changes in flights departed and arrived. Additionally, we look at the destinations where there were the most number of flights changes. 

## Data 

The dataset used for this update is from [Aviation Stack](https://aviationstack.com/documentation).This data is purchased through their API subscription. The data is being validated for Dubai against OAG aggregated dataset from April 2025-June 2025. 


In [125]:
import pandas as pd
from utils import *

Define a function to get a range of dates for the desirable date range

In [126]:
# Define empty dataset to concat all the arrivals and departures
departures = pd.DataFrame()
arrivals = pd.DataFrame()


In [127]:
import glob
from utils import *

arrivals = pd.DataFrame()

for file in glob.glob('../../data/aviation/arrivals_*.csv'):
    df = pd.read_csv(file)
    arrivals = pd.concat([arrivals,df])

#arrivals.drop(columns="Unnamed: 0", inplace=True)
arrivals.drop_duplicates(inplace=True)
arrivals.reset_index(drop=True, inplace=True)

for file in glob.glob('../../data/aviation/departures_*.csv'):
    df = pd.read_csv(file)
    departures = pd.concat([departures,df])

#departures.drop(columns="Unnamed: 0", inplace=True)
departures.drop_duplicates(inplace=True)
departures.reset_index(drop=True, inplace=True)

check_missing_dates(arrivals)
#check_missing_dates(departures)

All dates are present.


In [128]:
airports = ['DXB', 'DOH', 'AMM', 'JED', 'AUH', 'RUH', 'CAI', 'CMN']

In [129]:
# Check if the number of flights per day are 100. If they are exactly 100 we need to rerun the API to get the next 100 flight.
departures["flight_date"].value_counts()

flight_date
2026-02-14    6772
2026-01-03    6701
2026-01-17    6696
2026-02-07    6694
2026-01-31    6684
              ... 
2026-02-25    6010
2026-02-24    5951
2026-03-04    5950
2026-03-09    5928
2026-03-03    5870
Name: count, Length: 68, dtype: int64

In [130]:
# Check if the number of flights per day are 100. If they are exactly 100 we need to rerun the API to get the next 100 flight.
arrivals.sort_values(by="flight_date", ascending=False)["flight_date"].value_counts()

flight_date
2026-02-14    6775
2026-02-15    6767
2026-01-17    6750
2026-02-07    6733
2026-02-08    6725
              ... 
2026-02-25    6041
2026-03-04    6022
2026-02-24    6008
2026-03-09    5998
2026-03-03    5917
Name: count, Length: 68, dtype: int64

Functions to clean the database and explode columns

In [131]:
import ast


def safe_literal_eval(value):
    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return value
    return value


def explode(flights):
    flights["arrival"] = flights["arrival"].apply(safe_literal_eval)
    flights["departure"] = flights["departure"].apply(safe_literal_eval)

    fr1 = pd.json_normalize(flights["arrival"]).add_suffix("_arr")
    fr2 = pd.json_normalize(flights["departure"]).add_suffix("_dep")

    flights_exploded = pd.concat(
        [flights.drop(columns=["arrival", "departure"]), fr1, fr2], axis=1
    )

    return flights_exploded

In [132]:
%load_ext autoreload
%autoreload 2
from visuals import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [133]:
departures_exploded = explode(departures)
# departures_explode
# d = departures_exploded[~(departures_exploded['airport_arr']==departures_exploded['airport_dep'])]

In [134]:
check_missing_dates(departures)

All dates are present.


## Arrival Trends in Major Airports in MENAAP

In [135]:
beginning = departures["flight_date"].min()
end = departures["flight_date"].max()
print(f"Data is available from {beginning} to {end}")

Data is available from 2026-01-01 to 2026-03-09


Conduct a duplication check for flights. If the flight is taking off from the same place, to the same place at the same time and has two entries, it is a duplicate flight

In [136]:
before = departures_exploded.shape[0]
print(f"There were {before} flights before duplication check")
# check for duplicate flights i.e., flights scheduled to take off at the exact same time from the same place to the same destination
departures_exploded = departures_exploded.drop_duplicates(
    subset=["flight_date", "scheduled_arr", "iata_arr", "iata_dep", "scheduled_dep"]
)

after = departures_exploded.shape[0]
print(
    f"There are {after} flights after duplication check. {before-after} flights were duplicated"
)

There were 437345 flights before duplication check
There are 183667 flights after duplication check. 253678 flights were duplicated


### Flight Status Legend
- Scheduled: A flight that we have a schedule or flight plan for that hasn’t departed or has been canceled.
- Active: A flight that either left the gate or the runway and is on its way to its destination.
- Landed or Arrived: A flight that landed on the runway or arrived at the gate at the destination.
- Canceled: A flight that one or more data sources have indicated is canceled.
- Redirected: The flight is being redirected to another airport.
- Diverted: A flight that has landed or arrived at the gate of an airport where it wasn’t scheduled to arrive.
- Unknown: We were unable to detect the final arrival status.

In [137]:
# Test to see if any flight has more than one flight status assoctaed with it.
duplicate_status_test = (
    departures_exploded.groupby(
        ["flight_date", "scheduled_arr", "iata_arr", "iata_dep", "scheduled_dep"]
    )[["flight_status"]]
    .count()
    .reset_index()
)
duplicate_status_test[duplicate_status_test["flight_status"] > 1]

,flight_date,scheduled_arr,iata_arr,iata_dep,scheduled_dep,flight_status


In [138]:
# missing_dates

events = {
    "2024-06-09": "Airline data\nnot available",
}

### Daily Departures by Flight Status

In [139]:
departures_exploded["flight_date"] = pd.to_datetime(departures_exploded["flight_date"])

df = (
    departures_exploded.groupby(["flight_date", "flight_status", "iata_dep"])
    .size()
    .reset_index(name="count")
)

get_area_plot_by_airport(
    df,
    airports=airports,
    airport_col="iata_dep",
    title="Daily Departures by Airport",
    source_text="Source: Flight data from AviationStack",
)

alt.FacetChart(...)

### Weekly Departures by Flight Status

In [140]:
departures_exploded["flight_date"] = pd.to_datetime(departures_exploded["flight_date"])

df = (
    departures_exploded.groupby(
        [pd.Grouper(key="flight_date", freq="W"), "flight_status", "iata_dep"]
    )
    .size()
    .reset_index(name="count")
)

df = df[df['flight_date']<'2026-03-15']

get_area_plot_by_airport(
    df,
    airports=airports,
    airport_col="iata_dep",
    title="Weekly Departures by Airport",
    source_text="Source: Flight data from AviationStack",
    reindex_freq="W",
)

alt.FacetChart(...)

In [141]:
departures_exploded["airportcity"] = departures_exploded["iata_arr"].map(iata_mapping)
departures_poi = departures_exploded[departures_exploded["flight_date"] >= "2026-02-28"]

# Build a single DataFrame with top-20 destinations per departure airport per status category
records = []
for iata_dep, group in departures_poi.groupby("iata_dep"):
    for category, statuses in {
        "Most Changed": ["scheduled", "cancelled", "diverted"],
        "Most Unchanged": ["landed"],
    }.items():
        top20 = (
            group[group["flight_status"].isin(statuses)]["airportcity"]
            .value_counts()
            .head(20)
            .reset_index()
        )
        top20["category"] = category
        top20["iata_dep"] = iata_dep
        records.append(top20)

departures_top10 = pd.concat(records, ignore_index=True)

In [142]:
import altair as alt

airport_codes = sorted(departures_top10["iata_dep"].unique())

dropdown = alt.binding_select(options=airport_codes, name="Departure Airport ")
selection = alt.selection_point(fields=["iata_dep"], bind=dropdown, value=airport_codes[0])

colors = {"Most Changed": "#34A7F2", "Most Unchanged": "#FF9800"}

subplots = []
for category, color in colors.items():
    subset = departures_top10[departures_top10["category"] == category]

    chart = (
        alt.Chart(subset)
        .mark_bar(color=color, opacity=0.7)
        .encode(
            y=alt.Y("airportcity:N", sort="-x", title=None),
            x=alt.X("count:Q", title=None, axis=None),
            tooltip=[
                alt.Tooltip("airportcity:N", title="Destination"),
                alt.Tooltip("count:Q", title="Flights"),
            ],
        )
        .properties(width=150, height=400, title=category)
        .add_params(selection)
        .transform_filter(selection)
    )

    text = chart.mark_text(align="left", dx=3, fontSize=11, font="Open Sans").encode(
        text="count:Q"
    )

    subplots.append(chart + text)

final = (
    alt.hconcat(subplots[0], subplots[1])
    .properties(
        title=alt.Title(
            "Top 20 Destinations by Departure Status after 28th February 2026",
            subtitle="Source: AviationStack",
        )
    )
    .configure_title(font="Open Sans", subtitleFont="Open Sans")
    .configure_axis(labelFont="Open Sans", titleFont="Open Sans")
    .configure_legend(labelFont="Open Sans", titleFont="Open Sans")
    .configure_text(font="Open Sans")
)

final

alt.HConcatChart(...)

## Arrival Trends in Major Airports in MENAAP

In [143]:
check_missing_dates(arrivals)

All dates are present.


In [144]:
arrivals_exploded = explode(arrivals)

In [145]:
before = arrivals_exploded.shape[0]
print(f"There were {before} flights before duplication check")
# check for duplicate flights i.e., flights scheduled to take off at the exact same time from the same place to the same destination
arrivals_exploded = arrivals_exploded.drop_duplicates(
    subset=["flight_date", "scheduled_arr", "iata_arr", "iata_dep", "scheduled_dep"]
)

after = arrivals_exploded.shape[0]
print(
    f"There are {after} flights after duplication check. {before-after} flights were duplicated"
)

There were 440342 flights before duplication check
There are 184022 flights after duplication check. 256320 flights were duplicated


In [146]:
# Test to see if any flight has more than one flight status assoctaed with it.
duplicate_status_test = (
    arrivals_exploded.groupby(
        ["flight_date", "scheduled_arr", "iata_arr", "iata_dep", "scheduled_dep"]
    )[["flight_status"]]
    .count()
    .reset_index()
)
duplicate_status_test[duplicate_status_test["flight_status"] > 1]

,flight_date,scheduled_arr,iata_arr,iata_dep,scheduled_dep,flight_status


### Daily Number of Flights by Flight Status

In [147]:
arrivals_exploded["flight_date"] = pd.to_datetime(arrivals_exploded["flight_date"])

df = (
    arrivals_exploded.groupby(["flight_date", "flight_status", "iata_arr"])
    .size()
    .reset_index(name="count")
)

get_area_plot_by_airport(
    df,
    airports=['DXB', 'DOH', 'AMM', 'JED', 'AUH', 'RUH', 'CAI'],
    airport_col="iata_arr",
    title="Daily Arrivals by Airport",
    source_text="Source: Flight data from AviationStack",
)

alt.FacetChart(...)

### Weekly Arrivals by Flight Status

In [148]:
arrivals_exploded["flight_date"] = pd.to_datetime(arrivals_exploded["flight_date"])

df = (
    arrivals_exploded.groupby(
        [pd.Grouper(key="flight_date", freq="W"), "flight_status", "iata_arr"]
    )
    .size()
    .reset_index(name="count")
)

df = df[df['flight_date']<'2026-03-15']

get_area_plot_by_airport(
    df,
    airports=airports,
    airport_col="iata_arr",
    title="Weekly Arrivals by Airport",
    source_text="Source: Flight data from AviationStack",
    reindex_freq="W",
)

alt.FacetChart(...)

In [149]:
arrivals_exploded["airportcity"] = arrivals_exploded["iata_dep"].map(iata_mapping)
arrivals_poi = arrivals_exploded[arrivals_exploded["flight_date"] >= "2026-02-28"]

# Build a single DataFrame with top-10 origins per arrival airport per status category
records = []
for iata_arr, group in arrivals_poi.groupby("iata_arr"):
    for category, statuses in {
        "Most Changed": ["scheduled", "cancelled", "diverted"],
        "Most Unchanged": ["landed", "active"],
    }.items():
        top10 = (
            group[group["flight_status"].isin(statuses)]["airportcity"]
            .value_counts()
            .head(20)
            .reset_index()
        )
        top10["category"] = category
        top10["iata_arr"] = iata_arr
        records.append(top10)

arrivals_top10 = pd.concat(records, ignore_index=True)

### Origins with high and low disruptions

In [150]:
import altair as alt

airport_codes = sorted(arrivals_top10["iata_arr"].unique())

dropdown = alt.binding_select(options=airport_codes, name="Arrival Airport ")
selection = alt.selection_point(fields=["iata_arr"], bind=dropdown, value=airport_codes[0])

colors = {"Most Changed": "#34A7F2", "Most Unchanged": "#FF9800", }

subplots = []
for category, color in colors.items():
    subset = arrivals_top10[arrivals_top10["category"] == category]

    chart = (
        alt.Chart(subset)
        .mark_bar(color=color, opacity=0.7)
        .encode(
            y=alt.Y("airportcity:N", sort="-x", title=None),
            x=alt.X("count:Q", title=None, axis=None),
            tooltip=[
                alt.Tooltip("airportcity:N", title="Origin"),
                alt.Tooltip("count:Q", title="Flights"),
            ],
        )
        .properties(width=180, height=400, title=category)
        .add_params(selection)
        .transform_filter(selection)
    )

    text = chart.mark_text(align="left", dx=3, fontSize=11, font="Open Sans").encode(
        text="count:Q"
    )

    subplots.append(chart + text)

final = (
    alt.hconcat(subplots[0], subplots[1])
    .properties(
        title=alt.Title(
            "Top 20 Origins by Arrival Status after 28th February 2026",
            subtitle="Source: AviationStack | Changed flights = scheduled + cancelled + diverted | Unchanged flights = landed + active",
        )
    )
    .configure_title(font="Open Sans", subtitleFont="Open Sans")
    .configure_axis(labelFont="Open Sans", titleFont="Open Sans")
    .configure_legend(labelFont="Open Sans", titleFont="Open Sans")
    .configure_text(font="Open Sans")
)

final

alt.HConcatChart(...)

## Validation with OAG

In [210]:
oag_dxb_arr = pd.read_excel('../../data/oag/Arrivals_Dubai_JobId3639226.xlsx', skiprows=24)[0:3]
oag_dxb_arr

/Users/ssarva/Library/CloudStorage/OneDrive-WBG/Documents/MENA-FCV-economic-monitor/.venv/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Arr Airport Name,Frequency,Seats (Total),Time series
0,Dubai International,17131.0,4790086.0,2025-02
1,Dubai International,18381.0,5166344.0,2025-03
2,Dubai International,18979.0,5302576.0,2025-01


In [211]:
oag_dxb_dep = pd.read_excel('../../data/oag/Departures_Dubai_JobId3639227.xlsx', skiprows=24)[0:3]
oag_dxb_dep

/Users/ssarva/Library/CloudStorage/OneDrive-WBG/Documents/MENA-FCV-economic-monitor/.venv/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Dep Airport Name,Frequency,Seats (Total),Time series
0,Dubai International,17124.0,4787972.0,2025-02
1,Dubai International,18353.0,5157459.0,2025-03
2,Dubai International,18968.0,5299505.0,2025-01


In [213]:
# dxb_arr = pd.concat([
#     pd.read_csv('../../data/aviation/arrivals_DXB_2025-01-01_2025-03-31.csv')
# ])

dxb_dep = pd.concat([
    pd.read_csv('../../data/aviation/departures_DXB_2025-01-01_2025-03-31.csv')
])

In [214]:
# dxb_arr.reset_index(drop=True, inplace=True)
# dxb_arr_exploded = explode(dxb_arr)
# dxb_arr_exploded['flight_date'] = pd.to_datetime(dxb_arr_exploded['flight_date'])

dxb_dep.reset_index(drop=True, inplace=True)
dxb_dep_exploded = explode(dxb_dep)
dxb_dep_exploded['flight_date'] = pd.to_datetime(dxb_dep_exploded['flight_date'])

In [225]:
dxb_dep_exploded['flight_date'].unique()

<DatetimeArray>
['2025-03-11 00:00:00', '2025-03-12 00:00:00', '2025-03-13 00:00:00',
 '2025-03-14 00:00:00', '2025-03-15 00:00:00', '2025-03-16 00:00:00',
 '2025-03-17 00:00:00', '2025-03-18 00:00:00', '2025-03-19 00:00:00',
 '2025-03-20 00:00:00', '2025-03-21 00:00:00', '2025-03-22 00:00:00',
 '2025-03-23 00:00:00', '2025-03-24 00:00:00', '2025-03-25 00:00:00',
 '2025-03-26 00:00:00', '2025-03-27 00:00:00', '2025-03-28 00:00:00',
 '2025-03-29 00:00:00', '2025-03-30 00:00:00', '2025-03-31 00:00:00']
Length: 21, dtype: datetime64[ns]

In [215]:
# before = dxb_arr_exploded.shape[0]
# print(f"There were {before} flights before duplication check")
# # check for duplicate flights i.e., flights scheduled to take off at the exact same time from the same place to the same destination
# dxb_arr_exploded = dxb_arr_exploded.drop_duplicates(
#     subset=["flight_date", "scheduled_arr", "iata_arr", "iata_dep", "scheduled_dep"]
# )

# after = dxb_arr_exploded.shape[0]
# print(
#     f"There are {after} flights after duplication check. {before-after} flights were duplicated"
# )

before = dxb_dep_exploded.shape[0]
print(f"There were {before} flights before duplication check")
# check for duplicate flights i.e., flights scheduled to take off at the exact same time from the same place to the same destination
dxb_dep_exploded = dxb_dep_exploded.drop_duplicates(
    subset=["flight_date", "scheduled_arr", "iata_arr", "iata_dep", "scheduled_dep"]
)

after = dxb_dep_exploded.shape[0]
print(
    f"There are {after} flights after duplication check. {before-after} flights were duplicated"
)

There were 21733 flights before duplication check
There are 11893 flights after duplication check. 9840 flights were duplicated


In [199]:
dxb_arr_exploded[dxb_arr_exploded['iata_arr']=='DXB'].groupby(pd.Grouper(key='flight_date', freq='MS')).count()[['estimated_arr']]

,estimated_arr
flight_date,
2025-03-01,10995


In [222]:
dxb_dep_exploded.groupby(pd.Grouper(key='flight_date', freq='MS')).count()

,flight_status,airline,flight,aircraft,live,airport_arr,timezone_arr,iata_arr,icao_arr,terminal_arr,...,iata_dep,icao_dep,terminal_dep,gate_dep,delay_dep,scheduled_dep,estimated_dep,actual_dep,estimated_runway_dep,actual_runway_dep
flight_date,,,,,,,,,,,,,,,,,,,,,
2025-03-01,11893,11893,11893,7795,6,11569,11542,11893,11893,11503,...,11893,11893,11684,10726,11081,11893,11893,11027,11027,11027


In [ ]:
dxb_dep_exploded[dxb_dep_exploded['iata_arr']=='DXB'].groupby(pd.Grouper(key='flight_date', freq='MS')).count()[['estimated_dep']]